# Introduction to Big Data - Lab 02<br>Task 2.3: Find Pairs of Overlapping Shapes

## Student Information

Full Name: **NGUYỄN QUỐC THỊNH**\
Student ID: **22120347**

## Source Code

### Import Libraries, Files, and Setting Up Other Configurations

Import (third-party) `Python` libraries for use.

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, greatest, least, regexp_extract

Setup constant of paths of external processes.

In [ ]:
HDFS_URI = "hdfs://localhost:9000"

os.environ["JAVA_HOME"] = os.path.abspath("java-se-8u44-ri")
os.environ["SPARK_HOME"] = os.path.abspath("spark-3.5.5-bin-hadoop3")
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

INP_FILE_PATH = os.path.abspath("./shapes.parquet")
OUT_FILE_PATH = os.path.abspath(".")

Setup Spark session.

In [3]:
spark = SparkSession.builder\
    .appName("CountOverlappedRects") \
    .config("spark.hadoop.fs.defaultFS", HDFS_URI) \
    .getOrCreate()

print(spark.version)

25/04/11 22:48:50 WARN Utils: Your hostname, Luminous resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlp0s20f3)
25/04/11 22:48:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/11 22:48:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.5


Read the `shapes.parquet` and create the DataFrame `df`

In [4]:
df = spark.read \
          .option("inferSchema", "true") \
          .option("header", "true") \
          .parquet(f"file://{INP_FILE_PATH}")

df.show(truncate=False, vertical=False)
df.printSchema()

+--------+--------------------------------------------+
|shape_id|vertices                                    |
+--------+--------------------------------------------+
|Shape_0 |[[35, 28], [43, 28], [43, 32], [35, 32]]    |
|Shape_1 |[[44, 16], [53, 16], [53, 20], [44, 20]]    |
|Shape_2 |[[67, 84], [76, 84], [76, 92], [67, 92]]    |
|Shape_3 |[[29, 37], [37, 37], [37, 45], [29, 45]]    |
|Shape_4 |[[51, 79], [54, 79], [54, 88], [51, 88]]    |
|Shape_5 |[[55, 33], [62, 33], [62, 39], [55, 39]]    |
|Shape_6 |[[17, 88], [21, 88], [21, 97], [17, 97]]    |
|Shape_7 |[[1, 2], [10, 2], [10, 7], [1, 7]]          |
|Shape_8 |[[65, 14], [69, 14], [69, 20], [65, 20]]    |
|Shape_9 |[[12, 48], [19, 48], [19, 51], [12, 51]]    |
|Shape_10|[[97, 20], [100, 20], [100, 26], [97, 26]]  |
|Shape_11|[[7, 38], [15, 38], [15, 45], [7, 45]]      |
|Shape_12|[[69, 65], [78, 65], [78, 68], [69, 68]]    |
|Shape_13|[[35, 8], [42, 8], [42, 18], [35, 18]]      |
|Shape_14|[[96, 35], [100, 35], [100, 38], [96, 

Check whether any rectangle has edges that are not parallel to any of the axes.

In [5]:
not_parallel = df.select("*") \
    .where(
        (df['vertices'][0][0] != df['vertices'][3][0])
        | (df['vertices'][0][1] != df['vertices'][1][1])
        | (df['vertices'][1][0] != df['vertices'][2][0])
        | (df['vertices'][2][1] != df['vertices'][3][1])
    )

if not_parallel.isEmpty:
    print("All rectangles have their edges parallel to the axes.")
else:
    print("There exists a rectangle whose edges are not parallel to the axes.")

All rectangles have their edges parallel to the axes.


### Overlapping Detection Algorithm and Querying for Results

Create an inner-join DataFrame `joined` with all possible valid pairs of shapes (order doesn't matter)

In [6]:
df1 = df.alias("s1")
df2 = df.alias("s2")

joined = df1.crossJoin(df2) \
            .where(col("s1.shape_id") < col("s2.shape_id"))

joined.show()

+--------+--------------------+--------+--------------------+
|shape_id|            vertices|shape_id|            vertices|
+--------+--------------------+--------+--------------------+
| Shape_0|[[35, 28], [43, 2...| Shape_1|[[44, 16], [53, 1...|
| Shape_0|[[35, 28], [43, 2...| Shape_2|[[67, 84], [76, 8...|
| Shape_0|[[35, 28], [43, 2...| Shape_3|[[29, 37], [37, 3...|
| Shape_0|[[35, 28], [43, 2...| Shape_4|[[51, 79], [54, 7...|
| Shape_0|[[35, 28], [43, 2...| Shape_5|[[55, 33], [62, 3...|
| Shape_0|[[35, 28], [43, 2...| Shape_6|[[17, 88], [21, 8...|
| Shape_0|[[35, 28], [43, 2...| Shape_7|[[1, 2], [10, 2],...|
| Shape_0|[[35, 28], [43, 2...| Shape_8|[[65, 14], [69, 1...|
| Shape_0|[[35, 28], [43, 2...| Shape_9|[[12, 48], [19, 4...|
| Shape_0|[[35, 28], [43, 2...|Shape_10|[[97, 20], [100, ...|
| Shape_0|[[35, 28], [43, 2...|Shape_11|[[7, 38], [15, 38...|
| Shape_0|[[35, 28], [43, 2...|Shape_12|[[69, 65], [78, 6...|
| Shape_0|[[35, 28], [43, 2...|Shape_13|[[35, 8], [42, 8]...|
| Shape_

Do queries to find pairs of shapes that overlap with Spark.

In [7]:
result = joined.select(
    # Trasnform the "Shape_<id_number>" to just "<id_number>" to match the desired output
    regexp_extract(col('s1.shape_id'), '(\d+)', 1).alias('shape_1'),
    regexp_extract(col('s2.shape_id'), '(\d+)', 1).alias('shape_2'),

    # Get the length and width of each of the rectangles
    (col('s1.vertices')[1][0] - col('s1.vertices')[0][0]).alias('s1_length'),
    (col('s1.vertices')[2][1] - col('s1.vertices')[1][1]).alias('s1_width'),
    (col('s2.vertices')[1][0] - col('s2.vertices')[0][0]).alias('s2_length'),
    (col('s2.vertices')[2][1] - col('s2.vertices')[1][1]).alias('s2_width'),

    # Get the length and width of the rectangle that can contain both rectangles at the same time
    (greatest(col('s1.vertices')[1][0], col('s2.vertices')[1][0]) - least(col('s1.vertices')[0][0], col('s2.vertices')[0][0])).alias('s_length_range'),
    (greatest(col('s1.vertices')[2][1], col('s2.vertices')[2][1]) - least(col('s1.vertices')[1][1], col('s2.vertices')[1][1])).alias('s_width_range')
) \
.where(
    # IDEA: If the two rectanglaes are not overlapping, there will be gaps between them.
    # Therefore, the length and width of the big rectangle containing both rectangles at the same time 
    # will always be smaller than the sum of the two rectangles' lengths (and widths).
    (col('s_length_range') < col('s1_length') + col('s2_length'))
    & (col('s_width_range') < col('s1_width') + col('s2_width'))) \
.select('shape_1', 'shape_2')

result.show()

+-------+-------+
|shape_1|shape_2|
+-------+-------+
|      0|     41|
|      0|    143|
|      0|    198|
|      0|    430|
|      0|    473|
|      0|    490|
|      0|    496|
|      0|    605|
|      0|    637|
|      0|    729|
|      0|    818|
|      1|    118|
|      1|    160|
|      1|    186|
|      1|    220|
|      1|    247|
|      1|    256|
|      1|    377|
|      1|    444|
|      1|    809|
+-------+-------+
only showing top 20 rows



### Store the Results

Export the result to the `output.csv` file

In [8]:
result.toPandas().to_csv("output.csv", index=False)

End the Spark session

In [9]:
spark.sparkContext.stop()
spark.stop()